## Линейная регрессия

In [ ]:
import os
import sys

import matplotlib.pyplot as plt
import numpy as np

sys.path.append(os.path.abspath("../.."))
from students.kudryavcev.lesson2 import Exercise

rng = np.random.default_rng(777)
plt.style.use("seaborn-v0_8-whitegrid")

In [ ]:
def train_with_history(model, x: np.ndarray, y: np.ndarray, lr: float, n_iter: int, every: int = 20):
    history: list[tuple[int, float, float]] = []
    history.append((0, model.loss(x, y), model.metric(x, y)))
    for step in range(1, n_iter + 1):
        dw, db = model.grad(x, y)
        model.weights -= lr * dw
        model.bias -= lr * db
        if step % every == 0 or step == n_iter:
            history.append((step, model.loss(x, y), model.metric(x, y)))
    return history


def history_to_arrays(history):
    steps = np.array([row[0] for row in history])
    losses = np.array([row[1] for row in history])
    metrics = np.array([row[2] for row in history])
    return steps, losses, metrics

In [ ]:
n_samples = 150
x_lin_1d = rng.uniform(-3.0, 3.0, size=n_samples)
x_lin = x_lin_1d.reshape(-1, 1)
true_w = np.array([2.5])
true_b = -1.0
noise = rng.normal(scale=1.2, size=n_samples)
y_lin = x_lin[:, 0] * true_w[0] + true_b + noise

lin_model = Exercise.create_linear_model(num_features=1, rng=np.random.default_rng(1))
lin_history = train_with_history(lin_model, x_lin, y_lin, lr=0.05, n_iter=400, every=10)
lin_pred = lin_model.predict(x_lin)

print("learned weight:", lin_model.weights)
print("learned bias:", lin_model.bias)
print("final loss:", lin_history[-1][1])
print("final r2:", lin_history[-1][2])

In [ ]:
order = np.argsort(x_lin[:, 0])
x_sorted = x_lin[order, 0]
y_sorted_pred = lin_pred[order]
y_true_line = x_sorted * true_w[0] + true_b

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(x_lin[:, 0], y_lin, alpha=0.7, label="data")
axes[0].plot(x_sorted, y_true_line, color="green", linewidth=2, label="true line")
axes[0].plot(x_sorted, y_sorted_pred, color="red", linewidth=2, label="model prediction")
axes[0].set_title("Линейная регрессия: точки и прямая")
axes[0].set_xlabel("x")
axes[0].set_ylabel("y")
axes[0].legend()

axes[1].scatter(y_lin, lin_pred, alpha=0.7)
min_y = min(y_lin.min(), lin_pred.min())
max_y = max(y_lin.max(), lin_pred.max())
axes[1].plot([min_y, max_y], [min_y, max_y], color="black", linestyle="--")
axes[1].set_title("Истинные значения против предсказаний")
axes[1].set_xlabel("true y")
axes[1].set_ylabel("predicted y")

plt.tight_layout()
plt.show()

In [ ]:
steps, losses, r2_values = history_to_arrays(lin_history)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(steps, losses, marker="o")
axes[0].set_title("Loss по шагам")
axes[0].set_xlabel("step")
axes[0].set_ylabel("MSE loss")

axes[1].plot(steps, r2_values, marker="o", color="darkorange")
axes[1].set_title("R^2 по шагам")
axes[1].set_xlabel("step")
axes[1].set_ylabel("R^2")

plt.tight_layout()
plt.show()

In [ ]:
initial_loss = lin_history[0][1]
final_loss = lin_history[-1][1]
initial_r2 = lin_history[0][2]
final_r2 = lin_history[-1][2]

## Логистическая регрессия

In [ ]:
n_samples = 300
x_log = rng.normal(size=(n_samples, 2))
linear_score = 1.8 * x_log[:, 0] - 1.2 * x_log[:, 1] + 0.3
y_log = (linear_score > 0).astype(int)

log_model = Exercise.create_logistic_model(num_features=2, rng=np.random.default_rng(2))
log_history = train_with_history(log_model, x_log, y_log, lr=0.5, n_iter=800, every=20)
log_proba = log_model.predict(x_log)
log_pred = (log_proba >= 0.5).astype(int)

print("weights:", log_model.weights)
print("bias:", log_model.bias)
print("final loss:", log_history[-1][1])
print("final accuracy:", log_history[-1][2])

In [ ]:
x_min, x_max = x_log[:, 0].min() - 0.5, x_log[:, 0].max() + 0.5
y_min, y_max = x_log[:, 1].min() - 0.5, x_log[:, 1].max() + 0.5
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 300), np.linspace(y_min, y_max, 300))
grid = np.c_[xx.ravel(), yy.ravel()]
zz = log_model.predict(grid).reshape(xx.shape)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

contour = axes[0].contourf(xx, yy, zz, levels=np.linspace(0, 1, 11), cmap="RdBu", alpha=0.25)
axes[0].scatter(x_log[y_log == 0, 0], x_log[y_log == 0, 1], label="class 0", alpha=0.7)
axes[0].scatter(x_log[y_log == 1, 0], x_log[y_log == 1, 1], label="class 1", alpha=0.7)
axes[0].contour(xx, yy, zz, levels=[0.5], colors="black", linewidths=2)
axes[0].set_title("Классы и граница решения")
axes[0].set_xlabel("x1")
axes[0].set_ylabel("x2")
axes[0].legend()
fig.colorbar(contour, ax=axes[0], label="P(class=1)")

axes[1].scatter(log_proba, y_log + rng.normal(scale=0.02, size=len(y_log)), alpha=0.5)
axes[1].axvline(0.5, color="black", linestyle="--", label="threshold 0.5")
axes[1].set_title("Вероятности модели для объектов")
axes[1].set_xlabel("predicted probability")
axes[1].set_ylabel("true class")
axes[1].set_yticks([0, 1])
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
steps, losses, acc_values = history_to_arrays(log_history)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(steps, losses, marker="o")
axes[0].set_title("Log-loss по шагам")
axes[0].set_xlabel("step")
axes[0].set_ylabel("loss")

axes[1].plot(steps, acc_values, marker="o", color="darkorange")
axes[1].set_title("Accuracy по шагам")
axes[1].set_xlabel("step")
axes[1].set_ylabel("accuracy")

plt.tight_layout()
plt.show()

In [ ]:
initial_loss = log_history[0][1]
final_loss = log_history[-1][1]
initial_acc = log_history[0][2]
final_acc = log_history[-1][2]

### Еще примеры:

In [ ]:
n_samples_ext1 = 10
true_w_ext1 = np.array([90])
true_b_ext1 = -123.0
noise_scale_ext1 = 400

x_lin_1d_ext1 = rng.uniform(-3.0, 3.0, size=n_samples_ext1)
x_lin_ext1 = x_lin_1d_ext1.reshape(-1, 1)
noise_ext1 = rng.normal(scale=noise_scale_ext1, size=n_samples_ext1)
y_lin_ext1 = x_lin_ext1[:, 0] * true_w_ext1[0] + true_b_ext1 + noise_ext1

lin_model_ext1 = Exercise.create_linear_model(num_features=1, rng=np.random.default_rng(2))
lin_history_ext1 = train_with_history(lin_model_ext1, x_lin_ext1, y_lin_ext1, lr=0.05, n_iter=400, every=10)
lin_pred_ext1 = lin_model_ext1.predict(x_lin_ext1)

order_ext1 = np.argsort(x_lin_ext1[:, 0])
plt.figure(figsize=(7, 5))
plt.scatter(x_lin_ext1[:, 0], y_lin_ext1, alpha=0.7, label=f"Данные (шум={noise_scale_ext1})")
plt.plot(
    x_lin_ext1[order_ext1, 0],
    x_lin_ext1[order_ext1, 0] * true_w_ext1[0] + true_b_ext1,
    color="green",
    linewidth=2,
    label="Истинная прямая",
)
plt.plot(x_lin_ext1[order_ext1, 0], lin_pred_ext1[order_ext1], color="red", linewidth=2, label="Предсказание модели")
plt.title("Линейная регрессия")
plt.legend()
plt.show()

print("Финальная метрика R^2:", lin_history_ext1[-1][2])

In [ ]:
n_samples_ext2 = 2000
true_w_ext2 = np.array([-2.5])
true_b_ext2 = 1.0
noise_scale_ext2 = 1.5

x_lin_1d_ext2 = rng.uniform(-3.0, 3.0, size=n_samples_ext2)
x_lin_ext2 = x_lin_1d_ext2.reshape(-1, 1)
noise_ext2 = rng.normal(scale=noise_scale_ext2, size=n_samples_ext2)
y_lin_ext2 = x_lin_ext2[:, 0] * true_w_ext2[0] + true_b_ext2 + noise_ext2

lin_model_ext2 = Exercise.create_linear_model(num_features=1, rng=np.random.default_rng(3))
lin_history_ext2 = train_with_history(lin_model_ext2, x_lin_ext2, y_lin_ext2, lr=0.05, n_iter=400, every=10)
lin_pred_ext2 = lin_model_ext2.predict(x_lin_ext2)

order_ext2 = np.argsort(x_lin_ext2[:, 0])
plt.figure(figsize=(7, 5))
plt.scatter(x_lin_ext2[:, 0], y_lin_ext2, alpha=0.7, label="Данные")
plt.plot(
    x_lin_ext2[order_ext2, 0],
    x_lin_ext2[order_ext2, 0] * true_w_ext2[0] + true_b_ext2,
    color="green",
    linewidth=2,
    label="Истинная прямая",
)
plt.plot(x_lin_ext2[order_ext2, 0], lin_pred_ext2[order_ext2], color="red", linewidth=2, label="Предсказание модели")
plt.title("Линейная регрессия")
plt.legend()
plt.show()

print("Финальная метрика R^2:", lin_history_ext2[-1][2])

In [ ]:
n_samples_ext3 = 500
shift_1 = 0.0001

x_log_ext3_0 = rng.normal(loc=-shift_1, size=(n_samples_ext3 // 2, 2))
x_log_ext3_1 = rng.normal(loc=shift_1, size=(n_samples_ext3 // 2, 2))
x_log_ext3 = np.vstack([x_log_ext3_0, x_log_ext3_1])
y_log_ext3 = np.array([0] * (n_samples_ext3 // 2) + [1] * (n_samples_ext3 // 2))

log_model_ext3 = Exercise.create_logistic_model(num_features=2, rng=np.random.default_rng(2))
log_history_ext3 = train_with_history(log_model_ext3, x_log_ext3, y_log_ext3, lr=0.5, n_iter=800, every=20)
log_proba_ext3 = log_model_ext3.predict(x_log_ext3)
log_pred_ext3 = (log_proba_ext3 >= 0.5).astype(int)

plt.figure(figsize=(7, 5))
plt.scatter(x_log_ext3[:, 0], x_log_ext3[:, 1], c=log_pred_ext3, cmap="bwr", alpha=0.7, edgecolors="k")
plt.xlabel("Признак 1")
plt.ylabel("Признак 2")
plt.show()

print("Финальная метрика Accuracy:", log_history_ext3[-1][2])